# Capital Allocation II: What Estimation Error Costs

## 🎯 Learning Objectives

By the end of today you will be able to:

1. **Explain why an optimizer always wins in sample** — and why that is not evidence
2. **Decompose where a portfolio's Sharpe ratio actually comes from**
3. **Compare bet-sizing rules by the inputs they estimate**, and say which is exposed to what
4. **Measure estimation error with a test that has power** — and recognise one that does not
5. **Shrink the right input**, and say why shrinking the other one is destructive
6. **Tell a finding from a period**

## 📋 Today's Plan

1. [The in-sample illusion](#illusion)
2. [🔄 Where the Sharpe actually comes from](#where)
3. [The bet-sizing menu](#menu)
4. [How each rule is exposed](#exposed)
5. [Why the optimizer amplifies it](#why)
6. [What does the error cost?](#cost) — *🎯 prompt it*
7. [Give it a prior with the right shape](#prior)
8. [Finding, or period?](#period)
7. [🛠️ Hands-On](#ho1)
8. [🎯 Challenge](#challenge) — *homework*
9. [Key takeaways](#takeaways)

---

## 🛠️ Setup

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [11, 4]
import warnings; warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"

ff = pd.read_csv(f"{BASE}/ff_monthly.csv", index_col=0, parse_dates=True)
L  = pd.read_parquet(f"{BASE}/longshort_29.parquet")
F  = ff[['Mkt-RF','SMB','HML','RMW','CMA','UMD']].dropna()     # six factors
mkt = ff.loc[L.index, 'Mkt-RF']

sharpe = lambda x: x.mean() / x.std() * np.sqrt(12)
def W(m, S):                       # mean-variance weights, gross-normalised
    w = np.linalg.solve(S, m); return w / np.abs(w).sum()

# the hedged strategies from last lecture
H = pd.DataFrame({c: (lambda r: r.resid + r.params.iloc[0])(
        sm.OLS(L[c], sm.add_constant(mkt)).fit()) for c in L.columns})
print(f"{len(F)} months of six factors; {H.shape[1]} hedged strategies over {len(H)} months")

---

## 1 · The in-sample illusion <a id="illusion"></a>

Last lecture ended with a formula: $W^\star \propto \Sigma^{-1}\mu$. Every number
in it was estimated from the same data we then evaluated it on.

Here is what that buys, on the six Fama–French factors.

In [ ]:
#@title 🔒 Three portfolios, scored on the sample that built them
mu, S = F.mean().values, F.cov().values
print(f"  mean-variance    {sharpe(pd.Series(F.values @ W(mu, S))):.2f}")
print(f"  equal-weighted   {sharpe(F.mean(axis=1)):.2f}")
print(f"  best single      {max(sharpe(F[c]) for c in F):.2f}")

### It had to win

1.17 against 1.03. The optimizer beat equal weighting, and it was never in doubt:
$\Sigma^{-1}\mu$ **is** the portfolio with the highest Sharpe ratio in that
sample. It is the algebraic maximum. Reporting it as evidence that optimization
works is like reporting that the tallest person in the room is tall.

> **📌 Optimization is a form of in-sample regression.**
>
> It is fitting, and everything Lecture 8 said about fitting applies. The weights
> are chosen to make the past look good, and the more freedom you give them the
> better the past looks and the less it means.

Notice also how *small* the win is. Six factors, full freedom, forty-six years —
and the optimizer bought 0.14 of Sharpe ratio over dividing by six. Hold that
thought.

---

## 🔄 2 · Where the Sharpe actually comes from <a id="where"></a>

Forget the six factors. Take the thing you built last lecture: the alpha book
over 29 hedged strategies, weighted $w_i \propto \alpha_i/\sigma^2_{\epsilon,i}$,
in-sample Sharpe 2.26.

That formula uses two pieces of information — the alphas, and the residual
volatilities. So run a control that a scientist would insist on: **throw the
alphas away.** Weight by $1/\sigma^2_{\epsilon,i}$ alone, which knows nothing
whatsoever about which strategies are good.

> **🤔 Predict first.** How much worse should the no-information version be?

In [ ]:
#@title 🔒 Four weightings, in sample
al, sd = H.mean()*12, H.std()*np.sqrt(12)
rules = {'alpha book   a/sd^2': al/sd**2, 'NO ALPHA     1/sd^2': 1/sd**2,
         'risk parity  1/sd': 1/sd,       'equal        1/N': pd.Series(1., index=H.columns)}
for lab, w in rules.items():
    print(f"  {lab:24s} {sharpe((H*w).sum(axis=1)):.4f}")

### The version that knows nothing wins

**2.2924 against 2.2585.** Discarding the alphas — the entire point of the
exercise, the thing we spent all of Lecture 13 estimating — *improved* the
portfolio, in the sample the weights were fitted on.

That should be impossible. $\Sigma_\epsilon^{-1}\alpha$ is supposed to be the
in-sample maximum. It is not the maximum here because **the formula we used was
not $\Sigma_\epsilon^{-1}\alpha$** — it was the diagonal shortcut, which is only
the optimum if the residuals are uncorrelated. We showed last lecture that they
are not.

Now out of sample: estimate the weights on an expanding history, earn the next
month's return, never look forward.

In [ ]:
#@title 🔒 The same four, out of sample (expanding window, 120-month burn-in)
oos = {k: [] for k in rules}
for t in range(120, len(H)):
    Ht = H.iloc[:t]; a, s = Ht.mean(), Ht.std(); nxt = H.iloc[t]
    for k, w in [('alpha book   a/sd^2', a/s**2), ('NO ALPHA     1/sd^2', 1/s**2),
                 ('risk parity  1/sd', 1/s), ('equal        1/N', pd.Series(1., index=H.columns))]:
        oos[k].append((nxt*w).sum() / np.abs(w).sum())
for k, v in oos.items():
    print(f"  {k:24s} {sharpe(pd.Series(v, index=H.index[120:])):.4f}")

### Same ordering, out of sample

The no-alpha rule wins again — **2.07 against 1.85** — and equal weighting is
last. So the ranking is not a fluke of the fitting sample; it holds when nothing
is fitted.

Read the four numbers as a decomposition. Going from equal weighting to
$1/\sigma^2$ is worth about **+0.3**: that is the value of *risk information*,
which is easy to estimate. Going from there to $\alpha/\sigma^2$ is worth
**−0.2**: that is the value of *return information*, which is not.

> **📌 The alphas were the expensive input and the negative contributor.**
>
> Not because trailing alpha is uninformative — it does predict. Because the
> optimizer converts a noisy estimate into a *dispersed* set of weights, and the
> dispersion adds volatility faster than it adds return.

Which raises the obvious question. Why is the return information the part that
fails?

---

## 3 · The bet-sizing menu <a id="menu"></a>

Before fixing mean-variance, look at what you would use instead. Practitioners
have a standard menu, and **each entry is a different assumption about what you
know** — not a different level of sophistication.

| rule | weights | what it assumes | inputs it estimates |
|---|---|---|---|
| **mean-variance** | $\Sigma^{-1}\mu$ | you know both | μ **and** Σ |
| **minimum-variance** | $\Sigma^{-1}\mathbf{1}$ | all expected returns are equal | Σ only |
| **risk parity** | $1/\sigma_i$ | all *appraisal ratios* are equal | the diagonal of Σ |
| **proportional** | $\mu_i$ | you know the ranking, not the risk | μ only |
| **equal weight** | $1/N$ | you know the signs and nothing else | **nothing** |

Read the last column downward. That is the real ordering, and it is the one that
matters: **each rule is exposed to exactly the inputs it uses.** Minimum-variance
is not "mean-variance for cowards" — it is mean-variance under the explicit prior
that you cannot forecast returns, which for most assets over most samples is a
defensible thing to believe.

Risk parity deserves its own line. Setting $w_i \propto 1/\sigma_i$ means every
strategy contributes the same volatility to the book, which is exactly what
Lecture 13 §4 said to do **if all appraisal ratios were equal**. So risk parity
is not ignoring alpha; it is assuming you cannot distinguish alphas — a much
weaker claim than assuming they are zero.

> **📌 There is no rule here that is simply better. There is a rule that matches
> what you actually know.** The rest of this lecture is about finding out what
> that is.

---

## 4 · How each rule is exposed <a id="exposed"></a>

Out-of-sample horse races tell you which rule won on one path of history. They do
not tell you *why*, and they cannot be run before you have the history. So do the
thing that can: **hold the world fixed and vary the sample.**

### First: how much do the weights move?

Resample the months with replacement, refit each rule, and look at how far the
weights travel. A rule that estimates more inputs should move more.

In [ ]:
#@title 🔒 Bootstrap the sample 1,000 times, refit every rule
N = 6
RULES = {
 'mean-variance'  : lambda m, C: np.linalg.solve(C, m),
 'minimum-variance': lambda m, C: np.linalg.solve(C, np.ones(N)),
 'risk parity'    : lambda m, C: 1/np.sqrt(np.diag(C)),
 'equal 1/N'      : lambda m, C: np.ones(N),
}
nz = lambda w: w/np.abs(w).sum()
mu_a, S_a = F.mean().values*12, F.cov().values*12
base = {k: nz(f(mu_a, S_a)) for k, f in RULES.items()}

rng = np.random.default_rng(0); T = len(F); draws = {k: [] for k in RULES}
for _ in range(1000):
    X = F.values[rng.integers(0, T, T)]
    m, C = X.mean(0)*12, np.cov(X.T)*12
    for k, f in RULES.items(): draws[k].append(nz(f(m, C)))

print(f"  {'rule':18s}{'inputs used':>14s}{'avg weight sd':>16s}{'worst sign flip':>18s}")
for k, lab in [('mean-variance','mu and Sigma'), ('minimum-variance','Sigma'),
               ('risk parity','diag(Sigma)'), ('equal 1/N','none')]:
    D = np.array(draws[k])
    flip = np.mean(np.sign(D) != np.sign(base[k]), axis=0).max()
    print(f"  {k:18s}{lab:>14s}{D.std(0).mean():>16.3f}{flip:>18.0%}")

### The ordering is exactly the input list

Zero for equal weighting, which estimates nothing. **0.007** for risk parity,
which estimates only volatilities. **0.020** for minimum-variance, which
estimates the whole covariance matrix. **0.061** for mean-variance.

That last step is the one to stop on. Minimum-variance and mean-variance use the
*same* covariance matrix; the only difference between them is that mean-variance
also uses $\mu$. **Adding expected returns triples the weight uncertainty** — from
0.020 to 0.061 — and takes the worst sign-flip probability from 22% to **41%**.

This is the "means are harder to estimate than variances" claim, measured. You do
not have to take it on faith: the same bootstrap, the same data, one extra input.

### Second: how much true performance survives?

Weight instability is only bad if it costs you. So build a world where we know
the answer — draw returns from the estimated moments, so the true maximum Sharpe
ratio is a known number — then estimate weights from $T$ months of that world and
score them **against the truth**.

In [ ]:
#@title 🔒 A known world, four rules, four sample lengths
true_sr = np.sqrt(mu_a @ np.linalg.solve(S_a, mu_a))
C0 = np.linalg.cholesky(S_a/12)
print(f"  true maximum Sharpe ratio: {true_sr:.2f}   (what a perfect forecaster gets)\n")
print(f"  {'rule':18s}" + "".join(f"{f'{t//12}y':>9s}" for t in [60, 120, 240, 480]))
for k, f in RULES.items():
    row = f"  {k:18s}"
    for T_ in [60, 120, 240, 480]:
        keep = []
        for _ in range(400):
            X = rng.standard_normal((T_, N)) @ C0.T + mu_a/12
            w = f(X.mean(0)*12, np.cov(X.T)*12)
            keep.append((w @ mu_a)/np.sqrt(w @ S_a @ w)/true_sr)
        row += f"{np.median(keep):>9.0%}"
    print(row)

### A crossover, not a winner

Read across the rows and the whole debate resolves.

**Equal weighting is flat at 88%.** It never learns, so it never improves — and
it never errs, so it never collapses. It is a floor and a ceiling at once.

**Mean-variance goes 74% → 85% → 92% → 96%.** With five years of history it is
badly beaten by dividing by six. With twenty years it wins. **The crossover sits
somewhere between ten and twenty years of monthly data**, which is exactly the
range most people have — which is why this argument never ends.

**Risk parity is flat at 80%** and minimum-variance nearly flat at 84–87%: they
estimate little, so they gain little from more data and lose little from less.

> **📌 The choice of bet-sizing rule is a bet on how much data you have.**
>
> Not on which formula is cleverer. A rule that uses more inputs is buying a
> better answer with a worse estimate, and which side of that trade you want
> depends entirely on $T$.

One honest wrinkle before we move on. In this simulation the true covariance
matrix *is* the sample one — the world is stationary and correctly specified. Real
markets are not, which is why §6 will find that shrinking the covariance helps on
actual data even though this table says it should only add bias.

---

## 5 · Why the optimizer amplifies it <a id="why"></a>

§4 showed that adding $\mu$ triples the weight uncertainty. Here is the mechanism
that does the tripling, and it surprises people. Take two strategies with Sharpe ratios
$s_1 > s_2 > 0$ and correlation $\rho$. The optimal volatility allocations are

$$v_1 \propto s_1 - \rho s_2, \qquad v_2 \propto s_2 - \rho s_1$$

so the second weight goes **negative** whenever $s_2/s_1 < \rho$ — the optimizer
shorts an asset with a *positive* Sharpe ratio, using it as a hedge. That is not
a bug; it is the right answer. But it means the weights depend on small
differences between large, badly measured numbers.

In [ ]:
#@title 🔒 A real pair from our own 29
a, b = 'IdioVol3F', 'MaxRet'
p = H[[a, b]]; rho = p.corr().iloc[0,1]
w = W(p.mean().values, p.cov().values)
print(f"  {a}  Sharpe {sharpe(p[a]):.3f}")
print(f"  {b}     Sharpe {sharpe(p[b]):.3f}      correlation {rho:.3f}")
print(f"\n  s2/s1 = {sharpe(p[b])/sharpe(p[a]):.3f}  <  rho = {rho:.3f}   -> short the weaker one")
print(f"  optimal weights:  {a} {w[0]:+.3f}   {b} {w[1]:+.3f}")
print(f"  best single {max(sharpe(p[a]),sharpe(p[b])):.3f}  ->  optimal pair {sharpe((p*w).sum(axis=1)):.3f}")

### It shorts a 0.81-Sharpe strategy

`MaxRet` earns a Sharpe ratio of 0.81 on its own and the optimizer sells it,
because it correlates 0.92 with `IdioVol3F` and is slightly worse. As a hedge it
is more useful than it is as a holding.

The logic is right. The problem is what it does to the weights: they are set by
whether 0.860 is bigger or smaller than 0.922, and both of those are estimates.
Nudge either one and the position flips sign.

Which is exactly what §4's bootstrap measured.

In [ ]:
#@title 🔒 Perturb each expected return by one standard error
mu, S = F.mean().values, F.cov().values
base = np.sign(W(mu, S)); se = F.std().values/np.sqrt(len(F))
rng = np.random.default_rng(0)
flip = np.mean([np.sign(W(mu + rng.normal(0, se), S)) != base for _ in range(2000)], axis=0)
print("  probability the weight changes SIGN:")
for c, f in zip(F.columns, flip):
    print(f"    {c:8s} {f:5.0%}   {'  <- a coin flip' if f > 0.3 else ''}")

**HML flips 46% of the time.** The optimizer is not telling you to go long value.
It is telling you, in the only language it has, that it cannot tell — and the
language it has is a confident-looking number.

---

## 6 · What does the error cost? <a id="cost"></a>

We need a number for the damage. The obvious experiment: fit the weights on one
window, run them on the next, and compare against what was achievable in that
next window.

### 🎯 Prompt it — measure the cost of estimation error <a id="prompt"></a>

> **🤔 The question.** *"How much Sharpe ratio does my optimizer lose to
> estimation error?"*
>
> Write the prompt. The obvious experiment gives a large, confident, and
> meaningless number — and the way to find that out is to run it on something
> that has no estimation error at all.

In [ ]:
# === YOUR TURN ===
MY_PROMPT = """
                                    ← write your prompt here
"""

# ---- paste the AI's code below ----

In [ ]:
#@title 🔒 Check — the obvious experiment, and a placebo
Lw = 120; naive, placebo = [], []
w_full = W(F.mean().values, F.cov().values)          # sees the ENTIRE sample, past and future
for t in range(Lw, len(F)-Lw, 12):
    est, ev = F.iloc[t-Lw:t], F.iloc[t:t+Lw]
    best = sharpe(pd.Series(ev.values @ W(ev.mean().values, ev.cov().values)))
    naive.append(1 - sharpe(pd.Series(ev.values @ W(est.mean().values, est.cov().values)))/best)
    placebo.append(1 - sharpe(pd.Series(ev.values @ w_full))/best)
print(f"  estimated weights appear to lose   {np.median(naive):.0%}")
print(f"  CLAIRVOYANT weights appear to lose {np.median(placebo):.0%}   <- they have no error at all")

### The test has no power

The naive experiment says estimation error costs **46%** of the Sharpe ratio. Then
we ran the same experiment on a portfolio built from the *entire* sample —
weights that already know everything that will happen in the evaluation window —
and it "loses" **19%**.

A forecaster with literally zero forecast error is charged nineteen points of
forecast error. The measurement is broken, and it is broken for a reason worth
knowing: **the benchmark is the evaluation window's own in-sample maximum**, so
every portfolio ever constructed loses to it, including a perfect one.

> **⚠️ Whenever you benchmark against an in-sample optimum, run the placebo.**
> Score something that cannot be wrong. If it still looks wrong, your metric is
> measuring the benchmark, not the forecast.

So how much *does* it cost? **We already measured it in §4** — that is what the
known-world simulation was for. Read the mean-variance row again:

| history | 5 years | 10 years | 20 years | 40 years |
|---|---|---|---|---|
| **fraction of the achievable Sharpe you keep** | **74%** | 86% | 92% | 96% |

### The honest number, and it is not catastrophic

**26% at five years, 8% at twenty** — against the broken experiment's 46%. The
cost is real, it is large when your sample is short, and **it shrinks with data**,
which is the opposite of the fashionable claim that optimization simply does not
work.

Two things make this trustworthy where the first experiment was not. The
benchmark is the *true* optimum rather than a peeked-at one, so a perfect
forecaster scores 100% by construction. And the same simulation, asked a
different question in §4, produced a sensible ordering across rules — a broken
metric would not have.

The assumption doing the work is stationarity: returns drawn from one fixed
distribution we specified correctly. Real markets are not that kind, so treat
these as **lower bounds on the damage**.

---

## 7 · Give it a prior with the right shape <a id="prior"></a>

If the problem is a noisy input, the fix is to pull that input toward something
you believe before you have seen any data. That is **shrinkage**, and it is the
same idea as a constraint: both restrict where the answer can land.

There is a theorem worth knowing here. **A constraint can only reduce your Sharpe
ratio if your inputs are correct** — you are shrinking the feasible set, so the
optimum cannot improve. But if your inputs are *estimated*, a constraint can
raise it, because it stops the optimizer acting on noise. Long-only, position
caps, and 1/N are not admissions of defeat; they are priors.

The question is which input to shrink.

In [ ]:
#@title 🔒 Shrink the covariance, or shrink the means?
def oos_rule(fn, burn=120):
    r = [F.iloc[t].values @ fn(F.iloc[:t].mean().values, F.iloc[:t].cov().values)
         for t in range(burn, len(F))]
    return sharpe(pd.Series(r, index=F.index[burn:]))

print(f"  plain mean-variance          {oos_rule(lambda m,S: W(m,S)):.4f}")
print(f"  equal weighting              {oos_rule(lambda m,S: np.ones(6)/6):.4f}\n")
for a in [0.2, 0.4, 0.6, 0.8]:
    f = lambda m, S, a=a: W(m, (1-a)*S + a*np.trace(S)/6*np.eye(6))
    print(f"  shrink COVARIANCE  a={a:.1f}     {oos_rule(f):.4f}")
print()
for d in [0.5, 1.0]:
    f = lambda m, S, d=d: W((1-d)*m + d*m.mean(), S)
    print(f"  shrink MEANS       d={d:.1f}     {oos_rule(f):.4f}")

### Shrink the covariance. Never shrink the means toward each other.

Pulling the covariance matrix toward a scaled identity takes mean-variance from
**0.828 to 0.908**, and there is a genuine interior optimum around 60% — neither
the raw estimate nor the pure prior, but a blend. That is the classic
Ledoit–Wolf result, and an off-the-shelf implementation gets most of it with no
tuning at all.

Pulling the *means* toward their common average makes things monotonically
**worse**: 0.828, then 0.810, then 0.768. At full shrinkage every asset has the
same expected return and the portfolio is just minimum-variance.

That looks backwards — the means were the noisy input, so why does shrinking them
hurt? Because shrinking to a *common* mean does not remove noise, it removes the
only signal you had. The right prior for a covariance matrix is "everything is
similar and uncorrelated", which is a sensible thing to believe. The right prior
for expected returns is not "all equal"; it is "smaller than you think", which is
a different operation.

> **📌 Shrinkage helps when the prior has the right shape.** It is not a dial you
> turn up until things improve.

And the honest footnote: none of these reliably beats equal weighting, which
scores 0.917. The best shrunk optimizer lands at 0.908 — close, and behind.

---

## 8 · Finding, or period? <a id="period"></a>

That last sentence is the one people quote: *equal weighting beats the
optimizer.* Before you believe it, ask the question this course keeps asking.

In [ ]:
#@title 🔒 Split the sample in two
for lab, sl in [('1980-1999', slice('1980','1999')), ('2000-2026', slice('2000','2026'))]:
    D = F.loc[sl]; mv, eq = [], []
    for t in range(60, len(D)):
        Ht = D.iloc[:t]
        mv.append(D.iloc[t].values @ W(Ht.mean().values, Ht.cov().values))
        eq.append(D.iloc[t].values @ (np.ones(6)/6))
    print(f"  {lab}   mean-variance {sharpe(pd.Series(mv)):.3f}   "
          f"equal {sharpe(pd.Series(eq)):.3f}   gap {sharpe(pd.Series(eq))-sharpe(pd.Series(mv)):+.3f}")

### The result is a period

Equal weighting beat the optimizer by **+0.28** before 2000 and by **+0.01**
after. The famous result is a twentieth-century result. Over the last
twenty-six years the two are indistinguishable, and the full-sample gap of 0.089
has a bootstrap confidence interval that comfortably contains zero.

So the honest summary of this lecture is narrower and more useful than "the
optimizer fails":

> **📌 Mean-variance is fragile in exactly the way its inputs are fragile.** Feed
> it well-estimated risk and it earns its keep. Feed it poorly-estimated returns
> and it converts your uncertainty into confident-looking leverage. The fix is
> not to abandon it but to be honest about which input is which.

One thing this whole lecture ignored: **costs.** Every Sharpe ratio here is
gross. Mean-variance rebalances constantly and equal weighting barely trades, so
Lecture 11 says the gap moves further against the optimizer once you pay for it.
Any conclusion that turns on 0.05 of Sharpe ratio is one cost assumption from
reversing.

---

## 🛠️ Hands-On: Your Own Book <a id="ho1"></a>

> **🤔 Predict first.** For your group's strategy combined with the market, will
> the optimizer's weight or a 50/50 split do better out of sample?

In [ ]:
# === EDIT + YOUR TURN ===
MY_SIGNAL = "GP"      # ← your group's signal

pair = pd.concat([H[MY_SIGNAL].rename('mine'), mkt.rename('market')], axis=1).dropna()

# 1. In-sample optimal weights, and the Sharpe they produce.
w_is       = ____
sr_in      = ____

# 2. Out of sample: expanding window, 120-month burn-in, refit every month.
sr_oos     = ____

# 3. And a 50/50 equal-volatility split over the same out-of-sample months.
sr_equal   = ____

print(f"{MY_SIGNAL}:  in-sample {sr_in:.2f}   OOS optimizer {sr_oos:.2f}   OOS 50/50 {sr_equal:.2f}")

### Compare with the room

- **How much did you lose going from in-sample to out-of-sample?** Under 20% and
  you have a lot of data or a very stable pair.
- **Did the optimizer beat 50/50?** With two assets it usually does — the failure
  mode needs many correlated assets, not two.
- **Now the honest one.** You have run this once. What would you have to see to
  believe the answer rather than the sample?

---

## 🎯 Challenge: Audit the Book <a id="challenge"></a>

*Homework — due before the next class.*

You inherit the 29-strategy alpha book from Lecture 13 and are asked to defend
its construction.

### Q1 — The control

Out of sample, expanding window, 120-month burn-in, report the **Sharpe ratio of
the no-alpha rule** $w_i \propto 1/\sigma^2_{\epsilon,i}$ minus the Sharpe ratio
of the alpha book $w_i \propto \alpha_i/\sigma^2_{\epsilon,i}$.

A positive number means the alphas cost you.

> **📌 Required variable names:**
> ```python
> alpha_cost = ____   # SR(no-alpha) - SR(alpha book), out of sample
> ```

In [ ]:
# Your work here


alpha_cost = ____

print(f"discarding the alphas is worth {alpha_cost:+.3f} of Sharpe ratio")

### Q2 — The right amount of shrinkage

On the **six factors**, expanding window, 120-month burn-in, shrink the
covariance toward the scaled identity, $\;(1-a)\Sigma + a\,\frac{\mathrm{tr}\Sigma}{6}I$.
Search $a$ over `[0, 0.1, 0.2, ..., 1.0]` and report the $a$ with the highest
out-of-sample Sharpe ratio.

> **📌 Required variable names:**
> ```python
> best_a = ____   # the shrinkage intensity, a number in [0, 1]
> ```

In [ ]:
# Your work here


best_a = ____

print(f"best shrinkage intensity: {best_a:.1f}")

### Q3 — Does the shrinkage choice survive?

Now do what a real manager must: pick $a$ **without** seeing the future. Each
month, choose the $a$ that would have performed best on the history so far, then
use it for the next month.

Report the out-of-sample Sharpe ratio of that live procedure, and compare it with
Q2's number and with plain mean-variance (0.828).

> **📌 Required variable names:**
> ```python
> live_shrinkage_sr = ____   # OOS Sharpe when a is chosen live from history
> ```

In [ ]:
# Your work here


live_shrinkage_sr = ____

print(f"choosing a live: {live_shrinkage_sr:.3f}")

### Q4 — The memo

> **📝 Your task — maximum eight sentences.**
>
> Your firm runs the 29-strategy book. The head of research proposes replacing
> $\alpha/\sigma^2_\epsilon$ with plain $1/\sigma^2_\epsilon$ — throwing away the
> alpha estimates entirely — on the evidence in Q1.
>
> Say whether you agree and why, and be precise about what Q1 does and does not
> establish. Then use Q2 and Q3 together to answer the harder question: **you
> found a shrinkage intensity that improves things, and then found that choosing
> it honestly destroys the improvement. What, if anything, have you learned?**
> Finish by naming the one piece of evidence that would most change your mind
> about the alphas.

In [ ]:
MEMO = """
Write your memo here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ["alpha_cost", "best_a", "live_shrinkage_sr", "MEMO"]
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L13_CapitalAllocationII_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")

---

## 🧠 Key Takeaways <a id="takeaways"></a>

1. **An optimizer always wins in sample** — it is the algebraic maximum. That is
   not evidence of anything.

2. **Optimization is a form of in-sample regression.** Every warning from Lecture
   8 applies.

3. **Throwing the alphas away improved the book** — 2.29 against 2.26 in sample,
   2.07 against 1.85 out of sample. The expensive input was the negative
   contributor.

4. **Each rule is exposed to exactly the inputs it estimates.** Bootstrap weight
   dispersion: 1/N 0.000, risk parity 0.007, minimum-variance 0.020, mean-variance
   0.061. That last step is the price of using μ, and it triples the uncertainty.

5. **Choosing a rule is a bet on how much data you have.** Equal weighting is flat
   at 88% of the achievable Sharpe forever; mean-variance goes 74% → 96% from five
   years to forty. The crossover is at ten to twenty years — which is what everyone
   has, which is why the argument never ends.

5. **The optimizer shorts good assets** whenever $s_2/s_1 < \rho$. Correct, and it
   makes the weights turn on differences between badly measured numbers — HML's
   sign is a coin flip.

6. **Always run the placebo.** A clairvoyant portfolio "loses" 19% by the obvious
   metric. If something that cannot be wrong looks wrong, the metric is broken.

7. **The honest cost of estimation is 25% at five years and 8% at twenty** — real,
   and shrinking with data.

8. **Shrink the covariance, not the means.** 0.828 → 0.908 one way; 0.828 → 0.768
   the other. A prior helps only if it has the right shape.

9. **A constraint must hurt if your inputs are right and can help if they are
   estimated.** That is what 1/N is.

10. **"Equal weighting beats the optimizer" is a period, not a finding** — +0.28
    before 2000, +0.01 after.

---

### Next class

We have been choosing weights across strategies you already own. Next: where the
covariance matrix comes from in the first place, when you have three thousand
assets and two hundred months of data — and why every serious shop estimates it
with a factor model rather than a sample average.

---

## 📎 Appendix <a id="appendix"></a>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 📎 APPENDIX — the full shrinkage surface
# ═══════════════════════════════════════════════════════════════════════
print(f"  {'a':>5s}{'shrink covariance':>20s}{'shrink means':>16s}")
for a in np.arange(0, 1.01, 0.1):
    c = oos_rule(lambda m, S, a=a: W(m, (1-a)*S + a*np.trace(S)/6*np.eye(6)))
    d = oos_rule(lambda m, S, a=a: W((1-a)*m + a*m.mean(), S))
    print(f"  {a:5.1f}{c:>20.4f}{d:>16.4f}")
print(f"\n  equal weighting: {oos_rule(lambda m,S: np.ones(6)/6):.4f}")